# 03 - Feature analysis
Band power features, the EEGLearn-style spatial images, and a descriptive class difference. These are *descriptive* views of one participant, not significance tests.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline
from src.config import load_config
cfg = load_config()

In [ ]:
from src.data.dataset import load_recording, load_external_labels
from src.signal.epoching import build_trial_set
from src.features.spectral import band_power_from_config
from src.features.spatial import builder_from_config, ImageScaler, images_from_epochs
from src.visualization import spectra, topomaps
rec = load_recording(cfg, 1, "T")
ts = build_trial_set(rec, cfg, load_external_labels(cfg, 1, "T"))
feats = band_power_from_config(ts.X, ts.sfreq, cfg)
print("features (trials, bands, channels):", feats.shape)
fig = spectra.plot_psd_by_class(ts.X, ts.y, ts.sfreq, ts.ch_names, "C3", dict(cfg.features["bands_hz"]))
fig = spectra.plot_psd_by_class(ts.X, ts.y, ts.sfreq, ts.ch_names, "C4", dict(cfg.features["bands_hz"]))

In [ ]:
builder = builder_from_config(ts.ch_names, cfg)
imgs = images_from_epochs(ts.X, ts.sfreq, cfg, builder)
z = ImageScaler().fit_transform(imgs, builder.mask)
print("images (trials, feature channels, H, W):", z.shape, "| pixels inside electrode hull:", int(builder.mask.sum()), "of", builder.mask.size)
fig = topomaps.plot_electrode_layout(builder.xy, ts.ch_names, builder)
fig = topomaps.plot_class_difference_maps(z, ts.y, builder, list(cfg.features["bands_hz"]))

Remember: every coloured pixel is interpolated between electrodes; the black dots mark the template electrode positions.